# BBox Detection Difficulty Analysis
**FracAtlas — YOLOv8m vs YOLOv8s (AdamW)**

Classifies objects into **Easy / Moderate / Hard** levels via
mathematical thresholds on bbox parameters and compares models.

## 0. Imports

In [1]:
import os, sys, json, yaml, math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from tqdm import tqdm

plt.rcParams.update({'figure.figsize': (12, 6), 'figure.dpi': 100, 'savefig.bbox': 'tight'})
SAVE_DPI = 150
BASE_DIR = Path(r"D:\Project Medical Object Detection\FracAtlas")
OUTPUT_DIR = BASE_DIR / "notebooks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REF_W = REF_H = 1024  # training resolution
print("Libraries loaded.")

Libraries loaded.


## 1. Model Configs

In [2]:
YAML_PATH = BASE_DIR / "fracatlas_hand_oversampled.yaml"

MODELS = {
    "YOLOv8m-AdamW": {
        "run_dir": BASE_DIR / "runs-old-v8m-adamw" / "detect",
        "run_name": "fracatlas_yolov8m_adamw",
    },
    "YOLOv8s-AdamW": {
        "run_dir": BASE_DIR / "runs-old-v8s-adamw" / "detect",
        "run_name": "fracatlas_yolov8s_adamw",
    },
}

with open(YAML_PATH) as f:
    data_cfg = yaml.safe_load(f)
dataset_path = Path(data_cfg["path"])
test_img_dir = dataset_path / "test" / "images"
test_lbl_dir = dataset_path / "test" / "labels"

test_images = sorted(test_img_dir.iterdir())
print(f"Test images: {len(test_images)}")
for name, cfg in MODELS.items():
    pt = cfg["run_dir"] / cfg["run_name"] / "weights" / "best.pt"
    assert pt.exists(), f"{pt} not found"
    print(f"  {name}: {pt}")

Test images: 362
  YOLOv8m-AdamW: D:\Project Medical Object Detection\FracAtlas\runs-old-v8m-adamw\detect\fracatlas_yolov8m_adamw\weights\best.pt
  YOLOv8s-AdamW: D:\Project Medical Object Detection\FracAtlas\runs-old-v8s-adamw\detect\fracatlas_yolov8s_adamw\weights\best.pt


## 2. Helper Functions

In [3]:
def yolo_abs(xc, yc, w, h, iw, ih):
    return (xc - w/2)*iw, (yc - h/2)*ih, (xc + w/2)*iw, (yc + h/2)*ih

def area_abs(x1, y1, x2, y2):
    return max(0, x2-x1) * max(0, y2-y1)

def iou(b1, b2):
    xA, yA = max(b1[0], b2[0]), max(b1[1], b2[1])
    xB, yB = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    union = area_abs(*b1) + area_abs(*b2) - inter
    return inter / union if union > 0 else 0.0

def cls_area(a, t=None):
    te, tm = (96*96, 32*32) if t is None else (t[1], t[0])
    return "Easy" if a > te else ("Moderate" if a > tm else "Hard")

def cls_ar(w, h, t=3.0):
    r = max(w,h)/max(min(w,h),1)
    return "Easy" if r <= t else ("Moderate" if r <= t*2 else "Hard")

def cls_conf(c, t=None):
    te, tm = (0.7, 0.4) if t is None else (t[1], t[0])
    return "Easy" if c >= te else ("Moderate" if c >= tm else "Hard")

def composite(area, ar, conf):
    s = min(area / (REF_W*REF_H) * 10, 1.0) * 0.4
    s += max(0, 1.0 - (max(ar,1/ar) - 1.0) / 9.0) * 0.3
    s += conf * 0.3
    return ("Easy", s) if s >= 0.7 else (("Moderate", s) if s >= 0.4 else ("Hard", s))

## 3. Run Inference

In [4]:
def collect_preds(model, test_imgs, lbl_dir, conf_th=0.01):
    rows = []
    for img_path in tqdm(test_imgs, desc="Predicting"):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        with Image.open(img_path) as img:
            iw, ih = img.size
        gt_boxes = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                if not line.strip(): continue
                p = list(map(float, line.strip().split()))
                x1,y1,x2,y2 = yolo_abs(*p[1:5], iw, ih)
                gt_boxes.append([x1,y1,x2,y2,int(p[0])])
        preds = model.predict(str(img_path), conf=conf_th, iou=0.5, verbose=False)
        pred_boxes = []
        if preds and preds[0].boxes is not None:
            bx = preds[0].boxes
            for i in range(len(bx)):
                x1p,y1p,x2p,y2p = bx.xyxy[i].tolist()
                pred_boxes.append([x1p,y1p,x2p,y2p,float(bx.conf[i]),int(bx.cls[i])])
        n_gt = len(gt_boxes)
        matched = set()
        sx, sy = REF_W/iw, REF_H/ih
        for pb in pred_boxes:
            best_iou, best_idx = 0.0, -1
            for j, gb in enumerate(gt_boxes):
                if j in matched: continue
                iou_val = iou(pb[:4], gb[:4])
                if iou_val > best_iou:
                    best_iou, best_idx = iou_val, j
            is_tp = 1 if best_iou >= 0.5 and best_idx >= 0 else 0
            if is_tp: matched.add(best_idx)
            wr, hr = (pb[2]-pb[0])*sx, (pb[3]-pb[1])*sy
            rows.append({"image":img_path.name, "iw":iw, "ih":ih,
                "area_ref":wr*hr, "aspect":max(wr,hr)/max(min(wr,hr),1),
                "conf":pb[4], "iou":best_iou, "tp":is_tp, "n_gt":n_gt})
        for j, gb in enumerate(gt_boxes):
            if j in matched: continue
            wr, hr = (gb[2]-gb[0])*sx, (gb[3]-gb[1])*sy
            rows.append({"image":img_path.name, "iw":iw, "ih":ih,
                "area_ref":wr*hr, "aspect":max(wr,hr)/max(min(wr,hr),1),
                "conf":0.0, "iou":0.0, "tp":0, "n_gt":n_gt})
    return pd.DataFrame(rows)

all_results = {}
for name, cfg in MODELS.items():
    print(f"\nProcessing {name}...")
    df = collect_preds(YOLO(str(cfg["run_dir"]/cfg["run_name"]/"weights"/"best.pt")),
                       test_images, test_lbl_dir)
    all_results[name] = df
    tp, fn = int(df['tp'].sum()), int((df['conf']==0).sum())
    n_pred = len(df[df['conf']>0])
    print(f"  Rows:{len(df)}  TP:{tp}  FP:{n_pred-tp}  FN:{fn}  GT:{tp+fn}")
    print(f"  Mean IoU: {df[df['tp']==1]['iou'].mean() if tp>0 else 0:.4f}")


Processing YOLOv8m-AdamW...


Predicting: 100%|██████████| 362/362 [00:24<00:00, 14.50it/s]


  Rows:400  TP:215  FP:147  FN:38  GT:253
  Mean IoU: 0.7606

Processing YOLOv8s-AdamW...


Predicting: 100%|██████████| 362/362 [00:12<00:00, 29.35it/s]

  Rows:388  TP:210  FP:135  FN:43  GT:253
  Mean IoU: 0.7516


## 4. Assign Difficulty Labels

In [5]:
AREA_MOD, AREA_EASY = 32*32, 96*96
ASPECT_TH = 3.0
CONF_MOD, CONF_EASY = 0.4, 0.7

for _, df in all_results.items():
    df["da"] = df["area_ref"].apply(lambda a: cls_area(a, (AREA_MOD, AREA_EASY)))
    df["dasp"] = df["aspect"].apply(lambda a: cls_ar(a, ASPECT_TH))
    df["dconf"] = df["conf"].apply(lambda c: cls_conf(c, (CONF_MOD, CONF_EASY)))
    r = df.apply(lambda r: composite(r["area_ref"], r["aspect"], r["conf"]), axis=1, result_type="expand")
    df["dc"] = r[0]; df["dc_score"] = r[1]

print("Difficulty labels assigned.")
print(f"Area: Hard<={AREA_MOD} Mod<={AREA_EASY} Easy>{AREA_EASY}")
print(f"Aspect: Easy<={ASPECT_TH} Mod<={ASPECT_TH*2} Hard>{ASPECT_TH*2}")
print(f"Conf: Easy>={CONF_EASY} Mod>={CONF_MOD} Hard<{CONF_MOD}")
for name, df in all_results.items():
    print(f"\n{name}:\n{df['dc'].value_counts().to_string()}")

Difficulty labels assigned.
Area: Hard<=1024 Mod<=9216 Easy>9216
Aspect: Easy<=3.0 Mod<=6.0 Hard>6.0
Conf: Easy>=0.7 Mod>=0.4 Hard<0.4

YOLOv8m-AdamW:
dc
Moderate    212
Hard        188

YOLOv8s-AdamW:
dc
Moderate    208
Hard        180


## 5. Metrics by Difficulty

In [6]:
def metrics(df, col="dc"):
    rows = []
    for lv in ["Easy","Moderate","Hard"]:
        sub = df[df[col]==lv]
        if len(sub)==0: continue
        tp = int(sub["tp"].sum())
        preds = sub[sub["conf"]>0]
        fn = int((sub["conf"]==0).sum())
        n_p, n_g = len(preds), tp+fn
        fp = n_p - tp
        p = tp/max(n_p,1); r = tp/max(n_g,1)
        f1 = 2*p*r/max(p+r,1e-8)
        rows.append({"Level":lv,"GT":n_g,"Pred":n_p,"TP":tp,"FP":fp,"FN":fn,
            "Prec":p,"Rec":r,"F1":f1,
            "AvgIoU":sub[sub["tp"]==1]["iou"].mean() if tp>0 else 0.0,
            "AvgConf":preds["conf"].mean() if len(preds)>0 else 0.0})
    return pd.DataFrame(rows).set_index("Level")

met = {}
for name, df in all_results.items():
    met[name] = metrics(df, "dc")
    print(f"\n{name}:\n{met[name].to_string()}")


YOLOv8m-AdamW:
           GT  Pred   TP   FP  FN      Prec       Rec        F1    AvgIoU   AvgConf
Level                                                                              
Moderate  183   211  182   29   1  0.862559  0.994536  0.923858  0.769410  0.720566
Hard       70   151   33  118  37  0.218543  0.471429  0.298643  0.711788  0.076234

YOLOv8s-AdamW:
           GT  Pred   TP   FP  FN      Prec       Rec        F1    AvgIoU   AvgConf
Level                                                                              
Moderate  181   208  181   27   0  0.870192  1.000000  0.930591  0.756120  0.703696
Hard       72   137   29  108  43  0.211679  0.402778  0.277512  0.723201  0.066041


## 6. Single-Criterion Comparison

In [7]:
import csv

def load_map_from_results(run_dir, run_name):
    """Read best mAP50 and mAP50-95 from results.csv of a training run."""
    results_csv = run_dir / run_name / "results.csv"
    if not results_csv.exists():
        return None, None
    with open(results_csv, newline='') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    if not rows:
        return None, None
    # strip whitespace from keys
    rows = [{k.strip(): v.strip() for k, v in r.items()} for r in rows]
    # find key names (vary slightly between YOLO versions)
    map50_key   = next((k for k in rows[0] if 'map50' in k.lower() and '95' not in k.lower()), None)
    map5095_key = next((k for k in rows[0] if 'map50-95' in k.lower() or ('map' in k.lower() and '95' in k.lower())), None)
    if map50_key is None or map5095_key is None:
        return None, None
    # best epoch = row with highest mAP50
    best = max(rows, key=lambda r: float(r[map50_key]))
    return float(best[map50_key]), float(best[map5095_key])

# Pre-load mAP values for all models
model_maps = {}
for name, cfg in MODELS.items():
    m50, m5095 = load_map_from_results(cfg['run_dir'], cfg['run_name'])
    model_maps[name] = {'mAP50': m50, 'mAP50-95': m5095}
    status = f"mAP@50={m50:.4f}  mAP@50-95={m5095:.4f}" if m50 is not None else "results.csv not found"
    print(f"{name}: {status}")

print()
for col, title in [("da", "Area"), ("dasp", "Aspect"), ("dconf", "Conf")]:
    print(f"\n--- {title} ---")
    for name, df in all_results.items():
        print(f"{name}:")
        m = metrics(df, col)
        print(m[["GT", "Pred", "Prec", "Rec", "F1", "AvgIoU"]].to_string())
        # Tampilkan mAP@50 dan mAP@50-95 khusus YOLOv8s
        if 'YOLOv8s' in name:
            mp = model_maps[name]
            if mp['mAP50'] is not None:
                print(f"  [Overall val] mAP@50 = {mp['mAP50']:.4f} | mAP@50-95 = {mp['mAP50-95']:.4f}")
            else:
                print("  [Overall val] mAP@50 dan mAP@50-95: results.csv tidak ditemukan")
        print()

YOLOv8m-AdamW: mAP@50=0.8377  mAP@50-95=0.4289
YOLOv8s-AdamW: mAP@50=0.8472  mAP@50-95=0.4437


--- Area ---
YOLOv8m-AdamW:
           GT  Pred      Prec       Rec        F1    AvgIoU
Level                                                      
Easy       37    55  0.618182  0.918919  0.739130  0.754268
Moderate  211   304  0.592105  0.853081  0.699029  0.762361
Hard        5     3  0.333333  0.200000  0.250000  0.651514

YOLOv8s-AdamW:
           GT  Pred      Prec       Rec        F1    AvgIoU
Level                                                      
Easy       38    56  0.589286  0.868421  0.702128  0.751559
Moderate  210   288  0.614583  0.842857  0.710843  0.751577
Hard        5     1  0.000000  0.000000  0.000000  0.000000
  [Overall val] mAP@50 = 0.8472 | mAP@50-95 = 0.4437


--- Aspect ---
YOLOv8m-AdamW:
        GT  Pred      Prec       Rec        F1    AvgIoU
Level                                                   
Easy   253   362  0.593923  0.849802  0.699187  0.760565

YOL

## 7. Visualizations

In [8]:
CLR = {"Easy":"#2ecc71","Moderate":"#f39c12","Hard":"#e74c3c"}
ORD = ["Easy","Moderate","Hard"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, metric in enumerate(["Prec","Rec","F1"]):
    ax = axes[0,i]
    for name in all_results:
        m = met[name]
        ax.plot(ORD, [m.loc[lv][metric] if lv in m.index else 0 for lv in ORD],
                marker='o', linewidth=2.5, markersize=8, label=name)
    ax.set_title(f"{metric} by Difficulty"); ax.set_ylim([0,1.05])
    ax.grid(True, alpha=0.3); ax.legend()

for i,(col,title) in enumerate([("da","Area"),("dasp","Aspect"),("dconf","Conf")]):
    ax = axes[1,i]
    for name, df in all_results.items():
        m = metrics(df, col)
        ax.plot(ORD, [m.loc[lv]["F1"] if lv in m.index else 0 for lv in ORD],
                marker='s', linewidth=2.5, markersize=8, label=name)
    ax.set_title(f"F1 by {title}"); ax.set_ylim([0,1.05])
    ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"difficulty_analysis_comparison.png", dpi=SAVE_DPI)
plt.show()
print(f"Saved")

Saved


C:\Users\alema\AppData\Local\Temp\ipykernel_30488\1149378527.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, df) in zip(axes, all_results.items()):
    cnt = df["dc"].value_counts().reindex(ORD, fill_value=0)
    bars = ax.bar(ORD, cnt.values, color=[CLR[l] for l in ORD], edgecolor='white')
    for b, v in zip(bars, cnt.values):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(cnt.values)*0.02,
                str(v), ha='center', fontsize=11, fontweight='bold')
    ax.set_title(f"{name} Distribution"); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"difficulty_distribution.png", dpi=SAVE_DPI)
plt.show()

C:\Users\alema\AppData\Local\Temp\ipykernel_30488\1369357080.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (name, df) in zip(axes, all_results.items()):
    p = df[df["conf"]>0].copy()
    ax.scatter(p["area_ref"], p["conf"], c=p["tp"].map({1:"#2ecc71",0:"#e74c3c"}),
               alpha=0.5, s=15)
    ax.set_xlabel("Area (ref px^2)"); ax.set_ylabel("Confidence")
    ax.set_title(f"{name} Area vs Confidence"); ax.set_xscale('symlog')
    ax.axhline(CONF_EASY, color='gray', ls='--', alpha=0.5)
    ax.axhline(CONF_MOD, color='gray', ls=':', alpha=0.5)
    ax.axvline(AREA_EASY, color='green', ls='--', alpha=0.3)
    ax.axvline(AREA_MOD, color='orange', ls='--', alpha=0.3)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"area_vs_confidence_scatter.png", dpi=SAVE_DPI)
plt.show()

C:\Users\alema\AppData\Local\Temp\ipykernel_30488\1817189165.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, df) in zip(axes, all_results.items()):
    tp = df[(df["tp"]==1) & (df["dc"].isin(ORD))]
    data = [tp[tp["dc"]==lv]["iou"].values for lv in ORD]
    bp = ax.boxplot(data, tick_labels=ORD, patch_artist=True)
    for patch, c in zip(bp['boxes'], [CLR[l] for l in ORD]):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.set_title(f"{name} IoU by Difficulty"); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"iou_by_difficulty.png", dpi=SAVE_DPI)
plt.show()

C:\Users\alema\AppData\Local\Temp\ipykernel_30488\142859851.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Summary Table

In [12]:
rows = []
for name, df in all_results.items():
    tp = int(df["tp"].sum())
    fn = int((df["conf"]==0).sum())
    n_p = len(df[df["conf"]>0])
    fp = n_p - tp; n_g = tp + fn
    p = tp/max(n_p,1); r = tp/max(n_g,1)
    f1 = 2*p*r/max(p+r,1e-8)
    rows.append({"Model":name,"GT":n_g,"TP":tp,"FP":fp,"FN":fn,
        "Prec":f"{p:.4f}","Rec":f"{r:.4f}","F1":f"{f1:.4f}",
        "AvgIoU":f"{df[df['tp']==1]['iou'].mean() if tp>0 else 0:.4f}"})

s = pd.DataFrame(rows).set_index("Model")
print(s.to_string())
for name in all_results:
    print(f"\n{name}:\n{met[name][['GT','Pred','Prec','Rec','F1','AvgIoU']].to_string()}")
s.to_csv(OUTPUT_DIR/"difficulty_analysis_summary.csv")
print(f"\nSaved")

                GT   TP   FP  FN    Prec     Rec      F1  AvgIoU
Model                                                           
YOLOv8m-AdamW  253  215  147  38  0.5939  0.8498  0.6992  0.7606
YOLOv8s-AdamW  253  210  135  43  0.6087  0.8300  0.7023  0.7516

YOLOv8m-AdamW:
           GT  Pred      Prec       Rec        F1    AvgIoU
Level                                                      
Moderate  183   211  0.862559  0.994536  0.923858  0.769410
Hard       70   151  0.218543  0.471429  0.298643  0.711788

YOLOv8s-AdamW:
           GT  Pred      Prec       Rec        F1    AvgIoU
Level                                                      
Moderate  181   208  0.870192  1.000000  0.930591  0.756120
Hard       72   137  0.211679  0.402778  0.277512  0.723201

Saved


## 9. Threshold Config

| Param | Var | Default | Note |
|---|---|---|---|
| Area Hard | AREA_MOD | 1024 (32x32) | Below = Hard |
| Area Easy | AREA_EASY | 9216 (96x96) | Above = Easy |
| Aspect | ASPECT_TH | 3.0 | Easy<=3x Mod<=6x Hard>6x |
| Conf Easy | CONF_EASY | 0.7 | >=0.7 = Easy |
| Conf Mod | CONF_MOD | 0.4 | >=0.4 = Moderate |

Composite weights: Area 40%, Aspect 30%, Conf 30%. All areas scaled to 1024x1024 ref.